<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/27_final_results_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 27. Final results summary

This notebook collects the final numbers from notebooks 21-26 in one place: tables for the paper, four figures, and a short "supervisor's question -> our answer" list.

Evaluation protocol for notebooks 23-26: 24 stocks, forward-in-time split (train before 2021-12-15, 10-bar purge, test from 2022-01-01), F1 macro averaged over tickers. The "random baseline" is stratified guessing (it guesses labels randomly in the training proportions) and tells us what a model that knows nothing would score; with three classes this is about 0.33.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.options.display.float_format = '{:.3f}'.format

# --- Table 1: the six setups (notebooks 25 and 26), F1 macro vs random baseline
setups = pd.DataFrame([
    ('Events, pooled',           0.453, 0.359),
    ('Events, per-instrument',   0.318, 0.359),
    ('Bars, pooled',             0.353, 0.328),
    ('Bars, per-instrument',     0.366, 0.328),
    ('Conv1D on raw OHLC, pooled', 0.361, 0.328),
    ('RF on raw OHLC, pooled',   0.335, 0.328)],
    columns=['setup', 'f1_macro', 'random_baseline'])
setups['margin_over_random'] = setups.f1_macro - setups.random_baseline
print("Table 1: classification, 24 tickers, test from 2022"); print(setups.to_string(index=False))

# --- Table 2: trading on the same test events, 1.4 bps round trip (notebook 25)
trading = pd.DataFrame([
    ('Events, pooled',           563, 0.520,  0.347,  1.926, 17),
    ('Events, per-instrument',    58, 0.552,  1.115,  1.731, 10),
    ('Bars, pooled',             568, 0.493,  0.038,  0.210, 12),
    ('Bars, per-instrument',     582, 0.500,  0.166,  0.832, 13),
    ('No model (all events)',    885, 0.495, -0.024, -0.142, 15)],
    columns=['setup', 'trades', 'win_rate', 'mean_net_pct_per_trade', 't_stat', 'tickers_positive_of_24'])
print("\nTable 2: trading simulation"); print(trading.to_string(index=False))

# --- Table 3: walk-forward folds (notebook 24)
folds = pd.DataFrame([
    ('2018-2019', 0.463, 0.318), ('2020-2021', 0.451, 0.346),
    ('2022-2023', 0.518, 0.315), ('2024-2025', 0.446, 0.297)],
    columns=['test_block', 'f1_macro', 'random_baseline'])
print("\nTable 3: walk-forward, pooled event model"); print(folds.to_string(index=False))

# --- Table 4: training pool breadth on XOM / JPM holdouts (notebook 23)
pool = pd.DataFrame([
    ('XOM', 'dummy', 0.334, 0.187), ('XOM', '8 base', 0.465, 0.374),
    ('XOM', '+6 NASDAQ', 0.463, 0.394), ('XOM', '+INTC & value', 0.428, 0.395),
    ('JPM', 'dummy', 0.335, 0.160), ('JPM', '8 base', 0.441, 0.329),
    ('JPM', '+6 NASDAQ', 0.439, 0.353), ('JPM', '+INTC & value', 0.433, 0.373)],
    columns=['holdout', 'training_pool', 'f1_macro', 'f05_long_short'])
print("\nTable 4: breadth of the training pool"); print(pool.to_string(index=False))

# --- Table 5: F-beta on long/short classes (notebook 22)
fbeta = pd.DataFrame([('XOM', 0.506, 0.421, 0.483, 0.568), ('JPM', 0.433, 0.325, 0.354, 0.393)],
                     columns=['holdout', 'f1_macro', 'f0.5_ls', 'f1_ls', 'f2_ls'])
print("\nTable 5: F-beta on long/short classes"); print(fbeta.to_string(index=False))

# --- Table 6: cost sweep, total net return over the holdout history, % (notebook 22)
costs = pd.DataFrame({'cost_bps': [0, 1.4, 5, 10],
    'XOM ML': [-4.1, -5.3, -8.3, -12.4], 'XOM all events': [-33.1, -35.1, -40.2, -47.3],
    'JPM ML': [-26.1, -27.2, -29.9, -33.6], 'JPM all events': [-76.4, -78.7, -84.5, -92.7]})
print("\nTable 6: total net return vs round-trip cost (%), XOM and JPM holdouts, full history"); print(costs.to_string(index=False))

In [ ]:
blue, grey, orange = '#4C72B0', '#8C8C8C', '#DD8452'

# Figure 1: all setups vs random baseline
fig1, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(setups))
ax.barh(y, setups.f1_macro, color=blue)
ax.scatter(setups.random_baseline, y, color='k', marker='|', s=500, zorder=3, label='random baseline')
for i, v in enumerate(setups.f1_macro):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(setups.setup); ax.invert_yaxis()
ax.set_xlim(0.25, 0.50); ax.set_xlabel('Mean F1 macro over 24 tickers (test from 2022)')
ax.legend(loc='lower right'); ax.set_title('Classification: pooled event model is clearly best')
plt.tight_layout(); plt.show()

# Figure 2: walk-forward folds
fig2, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(folds)); w = 0.38
ax.bar(x - w/2, folds.f1_macro, w, color=blue, label='pooled event model')
ax.bar(x + w/2, folds.random_baseline, w, color=grey, label='random baseline')
ax.set_xticks(x); ax.set_xticklabels(folds.test_block); ax.set_ylabel('F1 macro')
ax.set_title('Forward-in-time test blocks'); ax.legend(); plt.tight_layout(); plt.show()

# Figure 3: trading, mean net return per trade
fig3, ax = plt.subplots(figsize=(8, 4))
colors = [blue if 'pooled' in s and 'Events' in s else grey for s in trading.setup]
ax.barh(trading.setup, trading.mean_net_pct_per_trade, color=colors)
for i, (m, t, n) in enumerate(zip(trading.mean_net_pct_per_trade, trading.t_stat, trading.trades)):
    ax.text(max(m, 0) + 0.02, i, f't={t:.2f}, n={n}', va='center', fontsize=9)
ax.axvline(0, color='k', lw=0.8); ax.invert_yaxis()
ax.set_xlabel('Mean net return per trade (%), 1.4 bps cost'); ax.set_xlim(-0.2, 1.7)
ax.set_title('Trading: only the pooled event model is clearly positive')
plt.tight_layout(); plt.show()

# Figure 4: cost sensitivity on XOM and JPM holdouts
fig4, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, tk in zip(axes, ['XOM', 'JPM']):
    ax.plot(costs.cost_bps, costs[f'{tk} ML'], marker='o', color=orange, label='ML')
    ax.plot(costs.cost_bps, costs[f'{tk} all events'], marker='o', color=blue, label='all events (no ML)')
    ax.axhline(0, color='k', lw=0.8); ax.set_title(tk); ax.set_xlabel('Round-trip cost (bps)')
axes[0].set_ylabel('Total net return over holdout history (%)'); axes[0].legend()
plt.tight_layout(); plt.show()

In [ ]:
for name, f in [('fig1_setups_f1', fig1), ('fig2_walkforward', fig2),
                ('fig3_trading', fig3), ('fig4_costs', fig4)]:
    f.savefig(f'{name}.png', dpi=200, bbox_inches='tight')
try:
    from google.colab import files
    for name in ['fig1_setups_f1', 'fig2_walkforward', 'fig3_trading', 'fig4_costs']:
        files.download(f'{name}.png')
except Exception as e:
    print("Download manually from the file panel:", e)

## The supervisor's questions and our answers

| Question | Answer (where) |
|---|---|
| Which features matter, and is it the same across stocks? | Short-horizon return and momentum features dominate on every stock; pattern flags matter on some. Rankings agree over all 54 features (Spearman about 0.85) but not over the top 20 (about 0.17). Feature selection gave no reliable gain, so we keep all features. (21) |
| Is F1 the right metric? Try F-beta. | On long/short classes F0.5 is below F2 on both holdouts (XOM 0.421 vs 0.568, JPM 0.325 vs 0.393): the model finds many real events but is often wrong when it opens a position. Precision is the weak side. (22) |
| Do transaction costs matter? | Not decisively. At the IBKR level (1.4 bps round trip) results barely move. In the 2022-2025 test the pooled event model's gross average (about +0.36% per trade) would only vanish at about 36 bps. On the XOM and JPM holdouts over their full history every strategy lost money even at zero cost. (22, 25) |
| Does more diverse training data help? | Not on F1 (XOM 0.465 -> 0.463 -> 0.428, JPM 0.441 -> 0.439 -> 0.433); a small possible gain in F0.5 on long/short. 8 to 14 stocks is enough. (23) |
| Does it work in different regimes and in the future? | Yes, above the random baseline in all four forward blocks (0.45-0.52 vs 0.30-0.35), including the COVID crash and the 2022 bear market. Weakest in sudden bear drops, best in sideways. (24) |
| One model or one per instrument? Events or every bar? | Events need pooling (pooled 0.453 vs per-instrument 0.318, pooled better in 21 of 24 tickers). Bars do not (per-instrument 0.366 vs pooled 0.353, within noise). The pooled event model is the best of the four. (25) |
| Does the strategy make money? | Only the pooled event model is clearly positive: +0.35% net per trade over 563 trades, 17 of 24 tickers positive, t = 1.93 (borderline). Bar-based models and no-model are indistinguishable from zero. (25) |
| Does a 1D convolutional network do better than the Random Forest? | No. Raw 20-bar OHLC gives F1 0.361 +- 0.009 for the Conv1D, on par with the RF with indicators (0.353-0.366) and barely above random (0.328). (26) |

## What the thesis can claim

1. Training the classifier on pattern events, pooled across many instruments, is the best setup we tested. It beats the random baseline by about 0.09 F1 (0.453 vs 0.359), is above baseline in every forward-in-time block, and gives a small positive trading result in 2022-2025.
2. Predictive skill is modest everywhere. Bar-by-bar models and a Conv1D on raw prices are only a few points above random.
3. Transaction costs are not the limiting factor; the signal is.
4. Results hold up across market regimes but weaken in sudden bear drops.

## What it cannot claim

- That the strategy is profitable in general. The pooled event result is borderline (t = 1.93), comes from a single 4-year period, and the trades overlap across correlated stocks. On the XOM and JPM holdouts over their full history (a different protocol, notebook 22), the strategy lost money.
- That differences of about 0.01 F1 between setups are real; seed-to-seed spread is up to 0.04 in places.
- Anything about intraday or other asset classes.

## Limits of the evidence

One seed in several comparisons, about 40 test events per ticker, correlated tickers, no confidence intervals or multiple-testing correction, and label definitions that differ between events and bars (so we compare margins over each setup's own random baseline).